# The Finite Difference That Gets Worse As h Shrinks

**Topic:** Numerical differentiation &nbsp;·&nbsp; **Difficulty:** Medium

Every gradient checker has the same surprise waiting: make `h` smaller and the
estimate improves, right up until it starts getting *worse*. Truncation error
falls as `h` shrinks, and roundoff error grows as you subtract two nearly equal
numbers and divide by something tiny.

Find the crossover empirically, and confirm that the central difference buys you
a whole extra order of accuracy for one extra function evaluation.

## The data (created in the next cell)

- `f` and `df`: a scalar function and its exact derivative.
- `x0 = 1.3`: the evaluation point.
- `hs`: 40 step sizes logarithmically spaced from `1e-1` down to `1e-14`.

## Your task

With **NumPy only**:

1. For each `h` in `hs`, compute the forward difference
   `(f(x0 + h) - f(x0)) / h` and the central difference
   `(f(x0 + h) - f(x0 - h)) / (2h)`.
2. Compute the absolute error of each against `df(x0)`.
3. Set `answers` to a dict with `"fwd_err"` and `"cen_err"` (length-40 arrays),
   `"best_h_central"` (the `h` minimizing the central error), and
   `"best_err_central"` (that minimum error).

## Evaluation

The error arrays are graded by their sum and one fixed random projection,
each to a relative tolerance of 2e-2. `best_h_central` and
`best_err_central` are scalars and are held to the tighter relative tolerance
of 1e-6. The evaluation cell also checks the central difference's best error is
smaller than the forward difference's best error.

In [ ]:
import numpy as np

def f(x):
    return np.sin(3 * x) * np.exp(-0.4 * x) + 0.5 * x ** 2

def df(x):
    return 3 * np.cos(3 * x) * np.exp(-0.4 * x) - 0.4 * np.sin(3 * x) * np.exp(-0.4 * x) + x

x0 = 1.3
hs = np.logspace(-1, -14, 40)
print(f"exact derivative at x0: {df(x0):.10f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use f, df, x0 and hs from the cell above, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("fwd_err", [40], 0.5641939554298919, -0.045166120019195025, 2e-02),
    ("cen_err", [40], 0.016334617935401724, -0.0006912167268746829, 2e-02),
    ("best_h_central", [], 4.641588833612782e-06, -7.771784198852709e-07, 1e-06),
    ("best_err_central", [], 1.8899354303769655e-11, -3.164470365899052e-12, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "fwd_err" in answers and "best_err_central" in answers:
    _fw = np.asarray(answers["fwd_err"], dtype=float).ravel()
    _win = bool(float(answers["best_err_central"]) < float(_fw[int(np.argmin(_fw))]))
    print(f"  {'central best err < fwd best err':<30} {'ok' if _win else 'wrong'}")
    _ok = _ok and _win

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Forward difference has truncation error `O(h)` and roundoff `O(eps/h)`, so it is
best near `h ~ sqrt(eps) ~ 1e-8`. Central difference cancels the quadratic term,
giving truncation `O(h^2)` against the same roundoff, so its optimum sits near
`h ~ eps^(1/3) ~ 6e-6` and is several orders more accurate. Below that, both
curves turn around and climb.

```python
import numpy as np

fwd = (f(x0 + hs) - f(x0)) / hs
cen = (f(x0 + hs) - f(x0 - hs)) / (2 * hs)

exact = df(x0)
fwd_err = np.abs(fwd - exact)
cen_err = np.abs(cen - exact)

j = int(np.argmin(cen_err))
answers = {
    "fwd_err": fwd_err,
    "cen_err": cen_err,
    "best_h_central": float(hs[j]),
    "best_err_central": float(cen_err[j]),
}
```